# Chapter 11 — Learning to Write in Recursive

## What this chapter is about

Chapter 10 taught you to **read** recursion and how the call stack works. 
This chapter teaches you to **write** it, using recognizable *categories* of 
problems, each with its own technique. Efficiency is deliberately ignored 
here. Recursion can be slow if used carelessly, and that is the topic of the 
next chapter.

## Category 1: Repeatedly Execute

The goal is to repeat a task (print a number, visit a directory, double a 
value). In these problems, the **last line of the function is a single, 
simple call to the function itself**, which makes the next iteration happen.

In [ ]:
def countdown(number):
    print(number)
    if number == 0:          # base case
        return
    countdown(number - 1)    # last line = the "next iteration" call

countdown(5)

## Trick 1: Passing extra parameters

A loop keeps track of its position with a variable like `index`. A recursive 
function has no loop variable, so **pass the state in as an extra parameter**. 
Each recursive call hands the updated value to the next call.

**In-place modification** means the function changes the original list instead 
of returning a new one. Afterward, the variable you passed in has changed.

**Default arguments** (`index=0`) keep the call clean, so callers don't have 
to pass the starting index themselves.

In [ ]:
def double_array(array, index=0):
    if index >= len(array):          # base case: walked past the end
        return
    array[index] *= 2                # modifies the ORIGINAL list in place
    double_array(array, index + 1)   # pass the incremented index forward

nums = [1, 2, 3, 4]
double_array(nums)
print(nums)    # [2, 4, 6, 8] -> the original list itself changed

## Category 2: Calculations

The goal is to **compute and return a value**, such as a sum, a count, or a 
factorial. Here recursion offers something loops don't: a different way of 
thinking, called **top-down**.

**Subproblem** = the same problem on a smaller input.
`factorial(6)` is `6 * factorial(5)`, so `factorial(5)` is its subproblem.

**Two strategies:**
- **Bottom-up**: build the answer starting from the smallest case (what a loop 
  naturally does). It can be done recursively with extra parameters, but that 
  adds little over a loop.
- **Top-down**: express the answer in terms of the answer to the subproblem. 
  Only recursion can do this, and it is the real reason recursion is powerful.

In [ ]:
# Bottom-up with recursion: needs extra parameters, clunky
def factorial_bottom_up(n, i=1, product=1):
    if i > n:
        return product
    return factorial_bottom_up(n, i + 1, product * i)

# Top-down: just rely on the subproblem
def factorial(n):
    if n == 1:
        return 1
    return n * factorial(n - 1)

print(factorial(6), factorial_bottom_up(6))   # 720 720

## The top-down thought process (the central idea of the chapter)

You get to "kick the problem down the road". You don't need to understand 
*how* the function solves the subproblem, only that it does. Three thoughts:

1. **Imagine the function you're writing is already implemented** by someone else.
2. **Identify the subproblem.** The first thing to try is usually the input 
   minus its first element/character (the "next-to-smallest" version).
3. **See what happens when you call the function on the subproblem**, and 
   figure out what small step turns that result into your answer.

Then add the base case last.

### Example: array sum

- Imagine `array_sum` already works.
- Subproblem: everything except the first element, so `[2,3,4,5]`.
- `array_sum([2,3,4,5])` gives `14`, so the answer is `1 + 14`.

In [ ]:
def array_sum(array):
    if len(array) == 1:                      # base case
        return array[0]
    return array[0] + array_sum(array[1:])   # first element + sum of the rest

array_sum([1, 2, 3, 4, 5])   # 15

### Example: string reversal

- Subproblem: the string minus its first character (`"bcde"` for `"abcde"`).
- `reverse("bcde")` gives `"edcb"`, so just put the first character at the end.

In [ ]:
def reverse(string):
    if len(string) == 1:                        # base case
        return string
    return reverse(string[1:]) + string[0]      # reversed rest + first char

reverse("abcde")   # 'edcba'

### Example: counting "x" (and a base-case simplification trick)

If the first character is `"x"`, the answer is `1 + count_x(rest)`, otherwise 
just `count_x(rest)`.

**Trick**: instead of using "one character left" as the base case (which forces 
two awkward sub-cases), use the **empty string / empty list** as the base case. 
Slicing past the end (`string[1:]` on a 1-character string) gives an empty 
string in Python, so the recursion naturally lands on it. This simplification 
works for the array sum and string reversal too.

In [ ]:
def count_x(string):
    if len(string) == 0:        # base case: empty string has zero x's
        return 0
    if string[0] == "x":
        return 1 + count_x(string[1:])
    return count_x(string[1:])

count_x("axbxcxd")   # 3

## The Staircase Problem — where top-down thinking really pays off

**Problem**: a staircase has N steps, and you can climb 1, 2, or 3 steps at a 
time. How many different paths reach the top?

Counting these by hand gets hard fast, but top-down makes it short. To reach 
step N, your *last move* came from step N-1, N-2, or N-3. Those cases don't 
overlap, and there are no other options. So:

`paths(n) = paths(n-1) + paths(n-2) + paths(n-3)`

**Base cases** are the tricky part, because the recursion reaches n of 0 or 
below. There are two ways to handle them:
- **Hardcode** them: `paths(<=0)=0`, `paths(1)=1`, `paths(2)=2`, `paths(3)=4`.
- **"Rig"** them: let `paths(0)=1` and `paths(negative)=0` so the arithmetic 
  produces the right answers for 2 and 3 on its own. This needs fewer lines but 
  is less intuitive.

In [ ]:
def number_of_paths(n):
    if n < 0:
        return 0
    if n == 0 or n == 1:
        return 1
    return number_of_paths(n - 1) + number_of_paths(n - 2) + number_of_paths(n - 3)

[number_of_paths(n) for n in range(1, 8)]
# [1, 2, 4, 7, 13, 24, 44]  -> matches the book's 1, 2, 4, 7 for 1-4 steps

**Heads-up (the next chapter's topic)**: this function recomputes the same 
subproblems over and over, exactly like the naive `fibonacci` you memoized in 
Week 2. Try `number_of_paths(30)` and notice the slowdown. The `@memoize` 
decorator you wrote would fix it directly.

## Anagram generation — recursion + loops together

**Problem**: return all anagrams (reorderings of all the characters) of a string.

**Top-down idea**: the subproblem is the string minus its first character. 
If you have all anagrams of `"bcd"`, you get all anagrams of `"abcd"` by 
**inserting `"a"` into every possible position** of each of them.

Notice that this solution uses a **loop inside the recursive function**. 
Recursion doesn't mean eliminating loops. Use each tool where it fits.

**Efficiency**: a string of length N has N! anagrams (3 characters give 
3·2·1 = 6), so this is **O(N!)**, factorial time, the slowest Big O category 
in the book. It can't be improved here, because the task is to *produce* all 
N! results.

In [ ]:
def anagrams_of(string):
    if len(string) == 1:                    # base case
        return [string]

    collection = []
    substring_anagrams = anagrams_of(string[1:])   # trust the subproblem

    for sub in substring_anagrams:
        for index in range(len(sub) + 1):          # every insertion position
            collection.append(sub[:index] + string[0] + sub[index:])

    return collection

anagrams_of("abc")
# ['abc', 'bac', 'bca', 'acb', 'cab', 'cba']  (same 6, different order than the book)

## Quick recipe for writing a recursive function

1. **Which category?** Repeating a task, or computing a value from a subproblem?
2. **Repeating a task** → make the last line the recursive call, and pass 
   extra parameters (index, accumulator) if you need to keep state.
3. **Computing** → imagine the function already works, pick the subproblem 
   (usually the input minus the first element), and combine the first element 
   with the subproblem's result.
4. **Last step: the base case.** Prefer the empty input (`0`, `""`, `[]`) over 
   "one element left" when it simplifies things.
5. **Check**: does every recursive call move closer to the base case?

## Exercises

1. **Total characters**: input is a list of strings, e.g. `["ab","c","def","ghij"]`. 
   Return the total number of characters across all strings (`10`). Subproblem 
   hint: the list minus its first string.
2. **Even numbers**: input is a list of numbers. Return a **new** list containing 
   only the even ones.
3. **Triangular numbers**: the sequence is 1, 3, 6, 10, 15, 21, 28... where the 
   Nth number is N plus the previous number. Input is a single integer `N` (not a 
   list). `triangular(7)` should return `28`.
4. **First "x"**: input is a string that definitely contains an `"x"`. Return the 
   index of the first one. Hint: this one needs the "extra parameter" trick from 
   the first category.
5. **Unique Paths**: input is two integers, `rows` and `columns`, describing a 
   grid. You start at the top-left square and must reach the bottom-right, moving 
   only **right** or **down**. Return the number of possible shortest paths. 
   Hint: the staircase problem is the closest cousin.